# Prepare Toronto ADA Wide Attributes

Join census data from `toronto-ada.csv` with ADA geometries from `toronto-ada.gpkg`
to produce a single GeoPackage and GeoJSON where each ADA feature has one attribute
column per census variable.

Values use `C10_RATE_TOTAL` (rate/percentage, `_pct` suffix) by default, except for
fields that are inherently counts or absolute values, which use `C1_COUNT_TOTAL` (`_count` suffix).

In [2]:
import os

import pandas as pd
import geopandas as gpd
from tqdm import tqdm

## Constants

In [2]:
# Characteristic IDs to include in the wide output.
# These are the active entries from CHARACTERISTIC_CODES in extract_census_ada.ipynb.
# Format: ID,  # <code> -- <full census characteristic name>
ACTIVE_IDS = [
    # Demographics - Population
    1,    # pop_2021 -- Population, 2021
    # 2,    # pop_2016 -- Population, 2016
    # 3,    # pop_pct_change -- Population percentage change, 2016 to 2021
    6,    # pop_density -- Population density per square kilometre

    # Age structure
    # 35,   # age_0_14 -- 0 to 14 years
    # 36,   # age_15_64 -- 15 to 64 years
    # 37,   # age_65_over -- 65 years and over
    # 39,   # age_mean -- Average age of the population
    40,   # age_median -- Median age of the population

    # Household composition
    57,   # pvt_house_avg_size -- Average household size

    # Income - Core measures
    115,  # income_aftertax_median -- Median after-tax income in 2020 among recipients ($)
    # 122,  # income_ei_recip -- Number of employment insurance benefits recipients
    # 124,  # income_covid_recip -- Number of COVID-19 emergency and recovery benefits recipients

    # Income - Poverty and distribution
    345,  # lim_at_prev -- Prevalence of low income based on LIM-AT (%)
    # 346,  # lim_at_0_17 -- LIM-AT prevalence, age 0 to 17 (%)
    # 347,  # lim_at_0_5 -- LIM-AT prevalence, age 0 to 5 (%)
    # 348,  # lim_at_18_64 -- LIM-AT prevalence, age 18 to 64 (%)
    # 349,  # lim_at_65_over -- LIM-AT prevalence, age 65 years and over (%)
    # 378,  # gini_measures -- Total - Inequality measures (population count used for Gini)
    379,  # gini_total_income -- Gini index on adjusted household total income
    # 380,  # gini_market_income -- Gini index on adjusted household market income
    # 381,  # gini_aftertax_income -- Gini index on adjusted household after-tax income
    # 382,  # gini_p90_p10 -- P90/P10 ratio on adjusted household after-tax income

    # Housing - Tenure and affordability
    # 1415, # housing_tenure_owner -- Owner
    1416, # housing_tenure_renter -- Renter
    # 1417, # housing_tenure_provided -- Dwelling provided by local government, First Nation or Indian band
    # 1418, # housing_condo_total -- Total - Occupied private dwellings by condominium status
    # 1466, # housing_shelter_under30 -- Spending less than 30% of income on shelter costs
    1467, # housing_shelter_30plus -- Spending 30% or more of income on shelter costs
    # 1468, # housing_shelter_30_100 -- Spending 30% to less than 100% of income on shelter costs

    # Housing - Core need
    1480, # housing_core_need_yes -- In core housing need (yes)
    # 1481, # housing_core_need_no -- Not in core housing need (no)
    # 1483, # housing_owner_mortgage_pct -- % of owner households with a mortgage
    # 1484, # housing_owner_shelter_30plus_pct -- % of owner households spending 30%+ on shelter costs
    # 1485, # housing_owner_core_need_pct -- % in core housing need

    # Citizenship
    1523, # citizen_canadian -- Canadian citizens
    # 1524, # citizen_canadian_under18 -- Canadian citizens aged under 18
    # 1525, # citizen_canadian_18over -- Canadian citizens aged 18 and over
    # 1526, # citizen_not_canadian -- Not Canadian citizens

    # Visible minorities
    1684, # visible_minority_yes -- Total visible minority population (yes)
    # 1697, # visible_minority_no -- Not a visible minority

    # Education - all ages 15+
    1999, # education_none -- No certificate, diploma or degree
    2000, # education_secondary -- High (secondary) school diploma or equivalency certificate
    # 2001, # education_postsec -- Postsecondary certificate, diploma or degree
    2008, # education_bachelor_higher -- Bachelor's degree or higher

    # Education - ages 25 to 64
    # 2015, # education_25_64_none -- No certificate, diploma or degree (age 25-64)
    # 2016, # education_25_64_secondary -- High (secondary) school diploma or equivalency certificate (age 25-64)
    # 2017, # education_25_64_postsec -- Postsecondary certificate, diploma or degree (age 25-64)
    # 2024, # education_25_64_bachelor_higher -- Bachelor's degree or higher (age 25-64)

    # Labour - Occupation
    2254, # labour_occupation_5_arts_culture -- Occupation 5 - Arts, culture, recreation and sport

    # Labour - Industry
    2278, # labour_industry_71_arts -- Industry 71 - Arts, entertainment and recreation
]

# Characteristic codes whose values are absolute counts or index values (not rates).
# These use C1_COUNT_TOTAL with a _count suffix; all others use C10_RATE_TOTAL with _pct.
COUNT_ONLY_CODES = {
    'pop_2021',               # Raw population count
    'pop_2016',               # Raw population count
    'age_mean',               # Mean age in years (not a rate)
    'age_median',             # Median age in years (not a rate)
    'pvt_house_avg_size',     # Average household size (not a rate)
    'income_aftertax_median', # Median after-tax income in $ (not a rate)
    'gini_total_income',      # Gini index value
    'gini_market_income',     # Gini index value
    'gini_aftertax_income',   # Gini index value
    'gini_p90_p10',           # P90/P10 ratio value
    'housing_condo_total',    # Raw count of condominium dwellings
}


In [3]:
# File paths
INPUT_CSV  = '../../data/census/ada/toronto-ada.csv'
INPUT_GPKG = '../../data/geo/toronto-ada.gpkg'
OUTPUT_DIR = '../../data/census/ada-wide'

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load and Pivot Census Data

In [4]:
# Load toronto-ada.csv and filter to active IDs only
df = pd.read_csv(INPUT_CSV, dtype={'GEO_NAME': str, 'CHARACTERISTIC_CODE': str}, encoding='latin')
df['CHARACTERISTIC_ID'] = pd.to_numeric(df['CHARACTERISTIC_ID'], errors='coerce')

df = df[df['CHARACTERISTIC_ID'].isin(ACTIVE_IDS)].copy()
print(f"Rows after filtering: {len(df)}  ({df['GEO_NAME'].nunique()} ADAs, {df['CHARACTERISTIC_ID'].nunique()} variables)")

Rows after filtering: 4743  (279 ADAs, 17 variables)


In [5]:
# Determine output column name and value for each row.
#   - COUNT_ONLY_CODES  → C1_COUNT_TOTAL,  suffix _count
#   - all others        → C10_RATE_TOTAL,  suffix _pct
#     (if C10 is null, fall back to C1_COUNT_TOTAL with _count)

df['C10_RATE_TOTAL'] = pd.to_numeric(df['C10_RATE_TOTAL'], errors='coerce')
df['C1_COUNT_TOTAL'] = pd.to_numeric(df['C1_COUNT_TOTAL'], errors='coerce')

is_count_only = df['CHARACTERISTIC_CODE'].isin(COUNT_ONLY_CODES)
c10_null      = df['C10_RATE_TOTAL'].isna()
use_count     = is_count_only | c10_null

df['col_name'] = df['CHARACTERISTIC_CODE'] + '_pct'
df['value']    = df['C10_RATE_TOTAL']

df.loc[use_count, 'col_name'] = df.loc[use_count, 'CHARACTERISTIC_CODE'] + '_count'
df.loc[use_count, 'value']    = df.loc[use_count, 'C1_COUNT_TOTAL']

# Pivot: one row per ADA, one column per variable
df_wide = df.pivot(index='GEO_NAME', columns='col_name', values='value').reset_index()
df_wide.columns.name = None

print(f"Wide table shape: {df_wide.shape}  ({len(df_wide)} ADAs x {df_wide.shape[1]-1} attributes)")
df_wide.head(2)

Wide table shape: (279, 18)  (279 ADAs x 17 attributes)


,GEO_NAME,age_median_count,citizen_canadian_pct,education_bachelor_higher_pct,education_none_pct,education_secondary_pct,gini_total_income_count,housing_core_need_yes_pct,housing_shelter_30plus_pct,housing_tenure_renter_pct,income_aftertax_median_count,labour_industry_71_arts_pct,labour_occupation_5_arts_culture_pct,lim_at_prev_pct,pop_2021_count,pop_density_pct,pvt_house_avg_size_count,visible_minority_yes_pct
0,35200001,37.2,88.4,26.6,20.5,29.0,0.241,11.6,27.2,16.4,33200.0,1.4,2.2,5.0,5547.0,278.6,3.6,94.9
1,35200002,38.0,86.8,25.9,21.1,31.2,0.244,9.3,28.3,12.0,32000.0,1.1,1.7,5.3,12479.0,1964.4,4.1,98.2


## Join with Geometry and Export

In [6]:
# Load ADA geometries — keep only ADAUID + geometry (drops Stats Canada metadata columns
# like ADANAME, PRUID, CSDUID etc. which inflates output file size without adding value)
gdf = gpd.read_file(INPUT_GPKG)
print(f"Geometry CRS: {gdf.crs}")
print(f"Geometry columns (all): {gdf.columns.tolist()}")

gdf = gdf[['ADAUID', 'geometry']]

# Join census wide table on ADAUID == GEO_NAME
gdf_wide = gdf.merge(df_wide, left_on='ADAUID', right_on='GEO_NAME', how='left')
gdf_wide = gdf_wide.drop(columns=['GEO_NAME'])  # drop the duplicate join key

print(f"\nJoined GDF shape: {gdf_wide.shape}")
print(f"Unmatched ADAs: {gdf_wide['pop_2021_count'].isna().sum()}")


Geometry CRS: EPSG:3347
Geometry columns (all): ['ADAUID', 'DGUID', 'LANDAREA', 'PRUID', 'geometry']

Joined GDF shape: (279, 19)
Unmatched ADAs: 0


In [7]:
# Save GeoPackage (original CRS) and GeoJSON (WGS84)
gpkg_path    = os.path.join(OUTPUT_DIR, 'toronto-ada-wide.gpkg')
geojson_path = os.path.join(OUTPUT_DIR, 'toronto-ada-wide.geojson')

print('Saving GeoPackage...')
gdf_wide.to_file(gpkg_path, driver='GPKG')

print('Saving GeoJSON (reprojected to WGS84)...')
# COORDINATE_PRECISION=5 gives ~1m accuracy (vs full double precision default),
# which is more than sufficient for ADA-level census boundaries and significantly
# reduces GeoJSON file size.
gdf_wide.to_crs('EPSG:4326').to_file(geojson_path, driver='GeoJSON', COORDINATE_PRECISION=5)

print(f'\nOutputs written to {OUTPUT_DIR}/')
print(f'  {os.path.basename(gpkg_path)}    ({os.path.getsize(gpkg_path)/1024:.0f} KB)')
print(f'  {os.path.basename(geojson_path)} ({os.path.getsize(geojson_path)/1024:.0f} KB)')


Saving GeoPackage...
Saving GeoJSON (reprojected to WGS84)...

Outputs written to ../../data/census/ada-wide/
  toronto-ada-wide.gpkg    (3412 KB)
  toronto-ada-wide.geojson (4707 KB)


---
## NOC / NAICS Creative Worker Data by Census Tract

Extract creative and cultural worker counts from the custom Stats Canada parquet
(`census2021_tract.parquet`) using DuckDB — the file is too large to load fully into memory.

**Geographic note:** This data is at the **census tract** level (1 227 tracts, Toronto CMA).
The ada-wide files above use **ADA** geography (279 ADAs, City of Toronto only).
To merge these two datasets a Stats Canada census-tract → ADA correspondence file is needed.
The output CSV uses `census_tract` as its key and follows the same column-naming convention
(`_count` / `_pct`) as the ada-wide files so the merge is straightforward once that crosswalk
is available.

In [ ]:
import duckdb

TRACT_PARQUET = '../../data/census/noc-naics/tac_census21/clean/census2021_tract.parquet'
NOC_OUTPUT_DIR = '../../data/census/noc-naics'
os.makedirs(NOC_OUTPUT_DIR, exist_ok=True)

# Each entry: (output_col_prefix, noc_group, naics_group, cip_group)
# All other demographic dimensions (immigrant_status, age_group, visible_minority,
# indigenous_identity, core_need, adl_type) are held at 'all' to get the total worker count.
SLICES = [
    ('pop_2021',                    'all',             'all',                            'all'),
    ('labour_creatives',            'creatives',       'all',                            'all'),
    ('labour_cultural_workers',     'cultural_workers','all',                            'all'),
    ('labour_cultural_industries',  'all',             'cultural_industries',             'all'),
    ('labour_independent_artists',  'all',             'independent_artists_arts_orgs',  'all'),
    ('labour_arts_major',           'all',             'all',                            'arts_major'),
]

con = duckdb.connect()
frames = []

for col_prefix, noc, naics, cip in SLICES:
    df_slice = con.execute(f"""
        SELECT
            census_tract,
            CASE WHEN value_suppressed THEN NULL ELSE value END AS value
        FROM read_parquet('{TRACT_PARQUET}')
        WHERE geography_level       = 'census_tract'
          AND row_kind              = 'population'
          AND noc_group             = '{noc}'
          AND naics_group           = '{naics}'
          AND cip_group             = '{cip}'
          AND immigrant_status      = 'all'
          AND age_group             = 'all'
          AND visible_minority      = 'all'
          AND indigenous_identity   = 'all'
          AND core_need             = 'all'
          AND adl_type              = 'all'
    """).fetchdf()
    df_slice = df_slice.rename(columns={'value': f'{col_prefix}_count'})
    frames.append(df_slice.set_index('census_tract'))

df_noc = pd.concat(frames, axis=1).reset_index()
df_noc.columns.name = None

# Compute _pct columns for each worker slice (count / total population × 100)
pop_col = 'pop_2021_count'
for col_prefix, _, _, _ in SLICES[1:]:
    df_noc[f'{col_prefix}_pct'] = (
        df_noc[f'{col_prefix}_count'] / df_noc[pop_col] * 100
    ).round(1)

print(f"Shape: {df_noc.shape}  ({len(df_noc)} census tracts × {df_noc.shape[1] - 1} attributes)")
print(f"Columns: {df_noc.columns.tolist()}")
print(f"Suppressed counts: { {c: int(df_noc[c].isna().sum()) for c in df_noc.columns if c != 'census_tract'} }")
df_noc.head(3)

In [ ]:
output_csv = os.path.join(NOC_OUTPUT_DIR, 'toronto-tract-noc-naics.csv')
df_noc.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}  ({os.path.getsize(output_csv) / 1024:.0f} KB)")
print(f"\nColumn summary:")
for col in df_noc.columns[1:]:
    non_null = df_noc[col].notna().sum()
    print(f"  {col:<42}  {non_null:>4} / {len(df_noc)} tracts non-null")

### Join with Census Tract Geometry and Export

Requires the Stats Canada 2021 census tract boundary file.
Download `lct_000b21a_e.zip` from the Statistics Canada 2021 Boundary Files page
and extract it to `data/geo/lct_000b21a_e/`.

In [ ]:
CT_SHP      = '../../data/geo/lct_000b21a_e/lct_000b21a_e.shp'
CSD_GPKG    = '../../data/geo/regions/GTA_CSD.gpkg'
ADA_GPKG    = '../../data/geo/toronto-ada.gpkg'
TORONTO_CMA = '535'   # first 3 chars of CTUID
TORONTO_CSD = '3520005'

# Load Toronto CSD boundary for spatial filtering
toronto_csd = gpd.read_file(CSD_GPKG)
toronto_csd = toronto_csd[toronto_csd['CSDUID'] == TORONTO_CSD][['geometry']]

# Load national CT boundaries, filter to Toronto CMA, then clip to Toronto CSD
# using centroid-within to avoid including boundary tracts from neighbouring municipalities.
gdf_ct = gpd.read_file(CT_SHP)
gdf_ct = gdf_ct[gdf_ct['CTUID'].str[:3] == TORONTO_CMA][['CTUID', 'geometry']].copy()
print(f"Toronto CMA tracts: {len(gdf_ct)}")

gdf_ct['_centroid'] = gdf_ct.geometry.centroid
gdf_ct = (
    gpd.sjoin(gdf_ct.set_geometry('_centroid'), toronto_csd, how='inner', predicate='within')
    .drop(columns=['_centroid', 'index_right'])
    .set_geometry('geometry')
)
print(f"Toronto CSD tracts:  {len(gdf_ct)}")

# Derive short-form tract code by stripping the 3-char CMA prefix from CTUID
gdf_ct['census_tract'] = gdf_ct['CTUID'].str[3:]

# Assign ADAUID to each tract via centroid-within-ADA spatial join.
# ADAs are larger than tracts so each tract centroid lands in exactly one ADA.
gdf_ada = gpd.read_file(ADA_GPKG)[['ADAUID', 'geometry']]
gdf_ct['_centroid'] = gdf_ct.geometry.centroid
gdf_ct = (
    gpd.sjoin(gdf_ct.set_geometry('_centroid'), gdf_ada, how='left', predicate='within')
    .drop(columns=['_centroid', 'index_right'])
    .set_geometry('geometry')
)
print(f"Tracts without ADAUID match: {gdf_ct['ADAUID'].isna().sum()}")

# Join NOC/NAICS attributes
df_noc_join = pd.read_csv(output_csv, dtype={'census_tract': str})
gdf_tract = gdf_ct.merge(df_noc_join, on='census_tract', how='left')

# Reorder: CTUID, census_tract, ADAUID, then data columns
cols = ['CTUID', 'census_tract', 'ADAUID'] + [c for c in gdf_tract.columns if c not in ('CTUID', 'census_tract', 'ADAUID', 'geometry')] + ['geometry']
gdf_tract = gdf_tract[cols]

print(f"\nJoined shape:      {gdf_tract.shape}")
print(f"Unmatched tracts:  {gdf_tract['pop_2021_count'].isna().sum()}")
gdf_tract.head(2)

In [ ]:
ADA_WIDE_DIR = '../../data/census/ada-wide'

gpkg_ct_path    = os.path.join(ADA_WIDE_DIR, 'toronto-tract-noc-naics.gpkg')
geojson_ct_path = os.path.join(ADA_WIDE_DIR, 'toronto-tract-noc-naics.geojson')

print('Saving GeoPackage...')
gdf_tract.to_file(gpkg_ct_path, driver='GPKG')

print('Saving GeoJSON (reprojected to WGS84)...')
gdf_tract.to_crs('EPSG:4326').to_file(geojson_ct_path, driver='GeoJSON', COORDINATE_PRECISION=5)

print(f'\nOutputs written to {ADA_WIDE_DIR}/')
print(f'  {os.path.basename(gpkg_ct_path)}    ({os.path.getsize(gpkg_ct_path)/1024:.0f} KB)')
print(f'  {os.path.basename(geojson_ct_path)} ({os.path.getsize(geojson_ct_path)/1024:.0f} KB)')

### Aggregate CT → ADA and Export

Sum `_count` columns from tract level up to ADA level, then recompute `_pct` from the
aggregated totals. Produces a GeoPackage and GeoJSON parallel to `toronto-ada-wide.*`.

In [ ]:
count_cols = [c for c in gdf_tract.columns if c.endswith('_count')]
pct_cols   = [c for c in gdf_tract.columns if c.endswith('_pct')]

# Sum counts per ADA (NaN-safe: skipna=True is the pandas default)
df_ada = gdf_tract[['ADAUID'] + count_cols].groupby('ADAUID', as_index=False)[count_cols].sum(min_count=1)

# Recompute _pct columns from aggregated counts
for col in pct_cols:
    worker_col = col.replace('_pct', '_count')
    df_ada[col] = (df_ada[worker_col] / df_ada['pop_2021_count'] * 100).round(1)

print(f"ADA aggregation shape: {df_ada.shape}  ({len(df_ada)} ADAs × {df_ada.shape[1]-1} attributes)")
print(f"Tracts per ADA — min: {gdf_tract.groupby('ADAUID').size().min()}, "
      f"max: {gdf_tract.groupby('ADAUID').size().max()}, "
      f"mean: {gdf_tract.groupby('ADAUID').size().mean():.1f}")
df_ada.head(3)

In [ ]:
gdf_ada_wide = gpd.read_file(ADA_GPKG)[['ADAUID', 'geometry']].merge(df_ada, on='ADAUID', how='left')

print(f"Joined ADA shape: {gdf_ada_wide.shape}")
print(f"Unmatched ADAs:   {gdf_ada_wide['pop_2021_count'].isna().sum()}")

gpkg_ada_path    = os.path.join(ADA_WIDE_DIR, 'toronto-ada-noc-naics.gpkg')
geojson_ada_path = os.path.join(ADA_WIDE_DIR, 'toronto-ada-noc-naics.geojson')

gdf_ada_wide.to_file(gpkg_ada_path, driver='GPKG')
gdf_ada_wide.to_crs('EPSG:4326').to_file(geojson_ada_path, driver='GeoJSON', COORDINATE_PRECISION=5)

print(f'\nOutputs written to {ADA_WIDE_DIR}/')
print(f'  {os.path.basename(gpkg_ada_path)}    ({os.path.getsize(gpkg_ada_path)/1024:.0f} KB)')
print(f'  {os.path.basename(geojson_ada_path)} ({os.path.getsize(geojson_ada_path)/1024:.0f} KB)')

---
## Future: Activity Data by Venue

When venue activity data (stops, visits, etc.) has been aggregated to the ADA level,
merge it here before saving.

In [9]:
# TODO: merge venue activity data
# activity_df = pd.read_csv('...')
# gdf_wide = gdf_wide.merge(activity_df, on='ADAUID', how='left')